# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/surajprai2111-ui/ML-assignment/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

**Classification, clustering, ranking, or scoring — which one, and why?**

I am framing my content performance project as a classification problem. The aim is to classify each content item into two groups: **declining** and **not declining**, based on its observed performance trend. This could help a content or SEO analyst decide which pages need closer review instead of checking all 30,000 records manually. I chose classification because the output is a category rather than a continuous number. This is a provisional framing; the next step would be to test whether the available content and search-related features can distinguish these groups reliably.

In [13]:
import pandas as pd
df=pd.read_csv("/content/content_refresh_anonymized.csv")
print("rows:",len(df))
print("columns;",len(df.columns))

rows: 30000
columns; 44


In [14]:
print(df["trend_direction"].value_counts(dropna=False))


trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64


## 2. Target or proxy

***What would you predict? Where does that label come from — observed outcome or a defined rule?***


Target variable: **is_declining**

My proposed target is is_declining, a binary label created from the existing trend_direction column. I will assign 1 when trend_direction is "down" and 0 for all other values.

This label comes from an observed trend in the dataset, so it is a **proxy for content needing attention**, not a direct measure of future performance. It does not prove that a page is poor or that updating it will improve clicks. Before using this target for a real predictive model, I would check how the trend label was created and make sure that information from the outcome period is not used as an input feature.Since is_declining is created directly from trend_direction, I would exclude trend_direction from the input features when training the model to avoid target leakage.

In [15]:
df["is_declining"] = (
    df["trend_direction"] == "down"
).astype(int)


print(df["is_declining"].value_counts())

print("\nPercentage distribution:")
print(
    (df["is_declining"].value_counts(normalize=True) * 100)
    .round(2)
)


is_declining
1    16262
0    13738
Name: count, dtype: int64

Percentage distribution:
is_declining
1    54.21
0    45.79
Name: proportion, dtype: float64


## 3. Success metric

***One metric you can defend. What number means 'good'?***

**Success metric: Recall for the declining class**

I would use recall for the declining class as the main success metric. Recall measures how many of the genuinely declining content items the model identifies correctly.

This matters because missing a page that is showing a decline could cause an analyst to overlook a potential improvement opportunity. For example, a recall of 80% would mean that the model identified 80 out of every 100 actual declining items in the evaluation data.

I would also check precision so that the model does not send too many pages for review unnecessarily. The final acceptable recall and precision values would depend on how much review time the content team has available.

The recall example below is illustrative. It demonstrates how the metric is calculated and is not a result from a trained model.

In [16]:
from sklearn.metrics import recall_score

# Example: actual labels and model predictions
actual = [1, 1, 1, 1, 0, 0, 0, 0, 1, 0]
predicted = [1, 1, 0, 1, 0, 1, 0, 0, 1, 0]

recall = recall_score(actual, predicted)

print("Recall for declining class:", round(recall, 2))

Recall for declining class: 0.8


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*



**Unit of analysis: One content item**

The unit of analysis is one content item, identified by content_id. Each row represents one content item and contains its search-related characteristics, content details, and measured performance over the available time periods.

For this task, the proposed target is is_declining. The input features could include information such as search_volume, competition, cpc, word_count, content_age_days, and other relevant content characteristics. I would need to remove identifiers and check for target leakage before training a model.

The dataset contains 30,000 rows, so each row represents one of those 30,000 content items rather than one individual click or impression.

In [19]:


# Select a useful slice for the ML framing
ml_slice = df[
    [
        "content_id",
        "search_volume",
        "competition",
        "cpc",
        "word_count",
        "content_age_days",
        "is_declining"
    ]
]

print("Shape of ML slice:", ml_slice.shape)

display(ml_slice.head())

print("\nTarget distribution:")
print(ml_slice["is_declining"].value_counts())

Shape of ML slice: (30000, 7)


,content_id,search_volume,competition,cpc,word_count,content_age_days,is_declining
0,content_304f48230142,10.0,0.67,2.05,3221.0,187,1
1,content_a1fb4e703a9e,90.0,0.01,0.05,2481.0,445,1
2,content_9aa793d4d895,0.0,0.00,0.00,3515.0,141,1
3,content_331d6c4de07b,10.0,0.00,0.00,NaN,463,0
4,content_d99b7a2d90ca,0.0,0.00,0.00,2803.0,263,1



Target distribution:
is_declining
1    16262
0    13738
Name: count, dtype: int64


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*


A fixed rule could flag every content item with low CTR or a large negative trend percentage. However, one threshold may not work equally well for all content items. A page with 10 impressions and zero clicks is different from a page with 10,000 impressions and 20 clicks. Content type, search volume, competition, content age, and other factors may also affect the observed performance.

A classification model could examine several features together and learn patterns associated with the proposed declining label. It may help prioritize content for review more flexibly than a single if-statement. However, ML is not automatically better: if a simple rule performs just as well, is easier to explain, and costs less to maintain, the rule may be the better choice. I would compare both approaches using the same evaluation data before deciding.

In [18]:

print("Content types:")
print(df["content_type"].value_counts(dropna=False))


print("\nDeclining rate by content type:")
print(
    df.groupby("content_type")["is_declining"]
      .mean()
      .mul(100)
      .round(2)
      .sort_values(ascending=False)
)

Content types:
content_type
keyword article       27207
feedly article         2096
comparison article      697
Name: count, dtype: int64

Declining rate by content type:
content_type
comparison article    57.25
keyword article       56.10
feedly article        28.67
Name: is_declining, dtype: float64


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.